# 📊 Week 5 – Customer Sales Analysis### Advanced Data Manipulation with Pandas**Project:** Customer purchasing patterns, top customers, and sales performance dashboard  **Dataset:** sales_data.csv | customer_data.csv | customer_churn.csv

## 1️⃣ Setup & Imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport seaborn as snsimport warningswarnings.filterwarnings('ignore')sns.set_style('whitegrid')plt.rcParams['figure.dpi'] = 110print("All libraries loaded ✅")

## 2️⃣ Data Loading & Exploration

In [ ]:
# Load all three datasetssales     = pd.read_csv('sales_data.csv', parse_dates=['order_date'])customers = pd.read_csv('customer_data.csv', parse_dates=['join_date'])churn     = pd.read_csv('customer_churn.csv')print(f"Sales Data      : {sales.shape[0]} rows × {sales.shape[1]} cols")print(f"Customer Data   : {customers.shape[0]} rows × {customers.shape[1]} cols")print(f"Churn Data      : {churn.shape[0]} rows × {churn.shape[1]} cols")

In [ ]:
# Preview sales datasales.head()

In [ ]:
# Column types and nullssales.info()

In [ ]:
# Statistical summarysales.describe().round(2)

In [ ]:
# Check missing values in customersprint("Missing values per column:")print(customers.isnull().sum()[customers.isnull().sum() > 0])

## 3️⃣ Data Cleaning & Preparation

In [ ]:
# Fill missing customer valuescustomers['age'].fillna(customers['age'].median(), inplace=True)customers['email'].fillna('unknown@example.com', inplace=True)# Verify no missing values remainprint("Missing values after cleaning:", customers.isnull().sum().sum())

In [ ]:
# String operations on text datacustomers['customer_name'] = customers['customer_name'].str.strip().str.title()customers['region']        = customers['region'].str.upper()customers['segment']       = customers['segment'].str.capitalize()# Extract date parts from order_datesales['year']    = sales['order_date'].dt.yearsales['month_n'] = sales['order_date'].dt.monthsales['day']     = sales['order_date'].dt.daysales['weekday'] = sales['order_date'].dt.day_name()print("Date columns extracted ✅")sales[['order_date','year','month_n','day','weekday']].head()

In [ ]:
# Merge customer details into salesmerged = sales.merge(    customers[['customer_id','customer_name','region','segment','age']],    on='customer_id',    how='left')print(f"Merged DataFrame: {merged.shape}")merged.head()

## 4️⃣ Aggregation Operations (3+ types)

In [ ]:
# AGGREGATION 1 – Monthly revenue summarymonth_order = ['January','February','March','April','May','June',               'July','August','September','October','November','December']monthly = (sales.groupby('month')['revenue']               .agg(total_revenue='sum', avg_order_value='mean', num_orders='count')               .reset_index())monthly['month'] = pd.Categorical(monthly['month'], categories=month_order, ordered=True)monthly = monthly.sort_values('month').reset_index(drop=True)print("Monthly Revenue Summary:")monthly.style.format({'total_revenue': '${:,.2f}', 'avg_order_value': '${:,.2f}'})

In [ ]:
# AGGREGATION 2 – Customer Lifetime Value (CLV)clv = (merged.groupby(['customer_id','customer_name'])             .agg(total_revenue=('revenue','sum'),                  num_orders=('order_id','count'),                  avg_order=('revenue','mean'))             .reset_index()             .sort_values('total_revenue', ascending=False))print("Top 10 Customers by CLV:")clv.head(10).style.format({'total_revenue':'${:,.2f}','avg_order':'${:,.2f}'})

In [ ]:
# AGGREGATION 3 – Product performanceprod_perf = (sales.groupby('product')                  .agg(total_revenue=('revenue','sum'),                       units_sold=('quantity','sum'),                       avg_unit_price=('unit_price','mean'),                       num_orders=('order_id','count'))                  .reset_index()                  .sort_values('total_revenue', ascending=False))print("Product Performance:")prod_perf.style.format({'total_revenue':'${:,.2f}','avg_unit_price':'${:,.2f}'})

## 5️⃣ Multi-Condition Filtering (AND / OR)

In [ ]:
# AND – High-value Electronics orders from Premium customersand_filter = merged[    (merged['revenue'] > 1000) &    (merged['category'] == 'Electronics') &    (merged['segment'] == 'Premium')]print(f"High-value Premium Electronics orders: {len(and_filter)}")and_filter[['customer_name','product','revenue','segment']].head()

In [ ]:
# OR – Orders from West region OR orders with >15% discountor_filter = merged[    (merged['region'] == 'WEST') |    (merged['discount'] > 0.15)]print(f"West-region OR high-discount orders: {len(or_filter)}")or_filter[['customer_name','region','discount','revenue']].head()

## 6️⃣ Pivot Tables

In [ ]:
# Revenue pivot: Category × Quarterpivot = pd.pivot_table(    sales,    values='revenue',    index='category',    columns='quarter',    aggfunc='sum',    fill_value=0)pivot['Total'] = pivot.sum(axis=1)print("Pivot Table – Revenue by Category × Quarter:")pivot.style.format('${:,.2f}').background_gradient(cmap='Blues', subset=pivot.columns[:-1])

## 7️⃣ Merging Datasets

In [ ]:
# Inner join sales with churn datasales_churn = merged.merge(    churn[['customer_id','churned','tenure_months']],    on='customer_id',    how='inner')# Compare revenue from churned vs retained customerschurn_revenue = (sales_churn.groupby('churned')['revenue']                             .agg(['sum','mean','count'])                             .reset_index())churn_revenue['churned'] = churn_revenue['churned'].map({0:'Retained', 1:'Churned'})churn_revenue.columns = ['Status','Total Revenue','Avg Revenue','Orders']print("Revenue – Churned vs Retained Customers:")churn_revenue.style.format({'Total Revenue':'${:,.2f}','Avg Revenue':'${:,.2f}'})

## 8️⃣ Visualisations Dashboard

In [ ]:
# Fig 1 – Monthly Revenue + Orders (twin-axis)fig, ax = plt.subplots(figsize=(12,5))ax.bar(monthly['month'].astype(str), monthly['total_revenue'], color='#2563EB', alpha=0.75, label='Revenue')ax2 = ax.twinx()ax2.plot(monthly['month'].astype(str), monthly['num_orders'], color='#EA580C',         marker='o', linewidth=2.5, label='Orders')ax.set_title('Monthly Revenue & Order Volume – 2023', fontsize=14, fontweight='bold')ax.set_ylabel('Revenue ($)'); ax2.set_ylabel('Number of Orders')ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))lines1,l1 = ax.get_legend_handles_labels(); lines2,l2 = ax2.get_legend_handles_labels()ax.legend(lines1+lines2, l1+l2, loc='upper left')plt.xticks(rotation=30, ha='right')plt.tight_layout(); plt.show()

In [ ]:
# Fig 2 – Top 10 Customers horizontal bartop10 = clv.head(10)fig, ax = plt.subplots(figsize=(10,6))colors = ['#EA580C'] + ['#2563EB']*9ax.barh(top10['customer_name'][::-1], top10['total_revenue'][::-1], color=colors[::-1])for bar, val in zip(ax.patches, top10['total_revenue'][::-1]):    ax.text(bar.get_width()+30, bar.get_y()+bar.get_height()/2, f'${val:,.0f}', va='center', fontsize=9)ax.set_title('Top 10 Customers by Revenue', fontsize=13, fontweight='bold')ax.set_xlabel('Total Revenue ($)')ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))plt.tight_layout(); plt.show()

In [ ]:
# Fig 3 – Category donut chartcat_rev = sales.groupby('category')['revenue'].sum().sort_values(ascending=False)palette = ['#2563EB','#16A34A','#EA580C','#7C3AED']fig, ax = plt.subplots(figsize=(7,6))wedges, texts, autotexts = ax.pie(cat_rev.values, labels=cat_rev.index,                                   autopct='%1.1f%%', startangle=140,                                   colors=palette, wedgeprops=dict(width=0.55))ax.set_title('Revenue by Category', fontsize=13, fontweight='bold')ax.text(0,0, f"Total\n${cat_rev.sum():,.0f}", ha='center', va='center',        fontsize=11, fontweight='bold')plt.tight_layout(); plt.show()

In [ ]:
# Fig 4 – Regional performanceregion_rev = merged.groupby('region')['revenue'].sum().sort_values(ascending=False)fig, ax = plt.subplots(figsize=(8,5))bars = ax.bar(region_rev.index, region_rev.values, color=palette)for bar, val in zip(bars, region_rev.values):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,            f'${val:,.0f}', ha='center', fontsize=9, fontweight='bold')ax.set_title('Revenue by Region', fontsize=13, fontweight='bold')ax.set_ylabel('Total Revenue ($)')ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))ax.set_ylim(0, region_rev.max()*1.15)plt.tight_layout(); plt.show()

In [ ]:
# Fig 5 – Pivot heatmapfig, ax = plt.subplots(figsize=(9,5))heat_data = pivot.drop(columns='Total')sns.heatmap(heat_data, annot=True, fmt='.0f', cmap='Blues', linewidths=0.5, ax=ax)ax.set_title('Revenue Heatmap: Category × Quarter', fontsize=13, fontweight='bold')plt.tight_layout(); plt.show()

In [ ]:
# Fig 6 – Segment analysisseg = merged.groupby('segment')['revenue'].agg(['sum','mean','count']).reset_index()seg.columns = ['segment','total','avg','count']fig, axes = plt.subplots(1,2, figsize=(11,4))axes[0].bar(seg['segment'], seg['total'], color=palette[:3])axes[0].set_title('Total Revenue by Segment', fontweight='bold')axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))axes[1].bar(seg['segment'], seg['avg'], color=palette[:3], alpha=0.8)axes[1].set_title('Avg Order Value by Segment', fontweight='bold')axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))plt.suptitle('Customer Segment Analysis', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

## 9️⃣ Key Findings & Business Insights

In [ ]:
total_rev = sales['revenue'].sum()top_cust  = clv.iloc[0]best_cat  = cat_rev.index[0]best_reg  = region_rev.index[0]print("=" * 55)print("  CUSTOMER SALES ANALYSIS REPORT – 2023")print("=" * 55)print(f"  Total Revenue      : ${total_rev:>12,.2f}")print(f"  Total Orders       : {len(sales):>12}")print(f"  Unique Customers   : {sales['customer_id'].nunique():>12}")print(f"  Avg Order Value    : ${sales['revenue'].mean():>12,.2f}")print(f"  Top Customer       : {top_cust['customer_name']} – ${top_cust['total_revenue']:,.2f}")print(f"  Best Category      : {best_cat}")print(f"  Best Region        : {best_reg}")print("=" * 55)print()print("RECOMMENDATIONS:")print("1. Expand Electronics inventory — 74%+ of revenue")print("2. Launch VIP programme for top 10 customers")print("3. Target underperforming regions with campaigns")print("4. Investigate Wearables Q4 gap — zero revenue")print("5. Run re-engagement campaign for churned customers")